In [1]:
# ── User Configuration ────────────────────────────────────────────────────────
symbolName      = "US500"   # Symbol name (must match broker exactly)
num_chunks      = 52 * 10   # Number of time chunks to fetch
weeks_per_chunk = 1         # Width of each chunk in weeks
period_str      = "M1"      # Bar period: M1 M2 M3 M4 M5 M10 M15 M30 H1 H4 H12 D1 W1 MN1

In [2]:
from ctrader_open_api import Client, Protobuf, TcpProtocol, Auth, EndPoints
from ctrader_open_api.messages.OpenApiCommonMessages_pb2 import *
from ctrader_open_api.messages.OpenApiMessages_pb2 import *
from ctrader_open_api.messages.OpenApiModelMessages_pb2 import *
from twisted.internet import reactor
import json
import datetime
import calendar
import keyring
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os

:0: UserWarning: You do not have a working installation of the service_identity module: 'No module named 'service_identity''.  Please install it from <https://pypi.python.org/pypi/service_identity> and make sure all of its dependencies are satisfied.  Without the service_identity module, Twisted can perform only rudimentary TLS client hostname verification.  Many valid certificate/hostname mappings may be rejected.


In [3]:
credentials = {
"ClientId": os.getenv("CLIENT_ID"),
"Secret": os.getenv("SECRET"),
"HostType": os.getenv("HOST_TYPE"),
"AccessToken": os.getenv("ACCESS_TOKEN"),
"AccountId": int(os.getenv("ACCOUNT_ID")),
}

host = EndPoints.PROTOBUF_LIVE_HOST if credentials["HostType"].lower() == "live" else EndPoints.PROTOBUF_DEMO_HOST
client = Client(host, EndPoints.PROTOBUF_PORT, TcpProtocol)

In [4]:
PERIOD_MAP = {
    "M1":  ProtoOATrendbarPeriod.M1,
    "M2":  ProtoOATrendbarPeriod.M2,
    "M3":  ProtoOATrendbarPeriod.M3,
    "M4":  ProtoOATrendbarPeriod.M4,
    "M5":  ProtoOATrendbarPeriod.M5,
    "M10": ProtoOATrendbarPeriod.M10,
    "M15": ProtoOATrendbarPeriod.M15,
    "M30": ProtoOATrendbarPeriod.M30,
    "H1":  ProtoOATrendbarPeriod.H1,
    "H4":  ProtoOATrendbarPeriod.H4,
    "H12": ProtoOATrendbarPeriod.H12,
    "D1":  ProtoOATrendbarPeriod.D1,
    "W1":  ProtoOATrendbarPeriod.W1,
    "MN1": ProtoOATrendbarPeriod.MN1,
}

if period_str not in PERIOD_MAP:
    raise ValueError(f"Unknown period '{period_str}'. Valid options: {list(PERIOD_MAP.keys())}")

bar_period  = PERIOD_MAP[period_str]
total_weeks = num_chunks * weeks_per_chunk
output_path = f"../../data/{symbolName}_{period_str}_{total_weeks}weeks.csv"
print(f"Output will be saved to: {output_path}")

Output will be saved to: ../../data/US500_M1_520weeks.csv


In [5]:
dailyBars = []

def transformTrendbar(trendbar):
    openTime   = datetime.datetime.fromtimestamp(trendbar.utcTimestampInMinutes * 60, datetime.timezone.utc)
    openPrice  = (trendbar.low + trendbar.deltaOpen)  / 100000.0
    highPrice  = (trendbar.low + trendbar.deltaHigh)  / 100000.0
    lowPrice   =  trendbar.low                        / 100000.0
    closePrice = (trendbar.low + trendbar.deltaClose) / 100000.0
    return [openTime, openPrice, highPrice, lowPrice, closePrice, trendbar.volume]

In [6]:
def symbolsResponseCallback(result):
    print("\nSymbols received")
    symbols = Protobuf.extract(result)
    symbolsFilterResult = list(filter(lambda s: s.symbolName == symbolName, symbols.symbol))
    if len(symbolsFilterResult) == 0:
        raise Exception(f"No symbol matches '{symbolName}'")
    elif len(symbolsFilterResult) > 1:
        raise Exception(f"Multiple symbols match '{symbolName}': {symbolsFilterResult}")
    symbol = symbolsFilterResult[0]

    now = datetime.datetime.utcnow()
    requests = []
    for i in range(num_chunks):
        to_time   = now - datetime.timedelta(weeks=weeks_per_chunk * i)
        from_time = to_time - datetime.timedelta(weeks=weeks_per_chunk)
        request = ProtoOAGetTrendbarsReq()
        request.symbolId            = symbol.symbolId
        request.ctidTraderAccountId = credentials["AccountId"]
        request.period              = bar_period
        request.fromTimestamp       = int(calendar.timegm(from_time.utctimetuple())) * 1000
        request.toTimestamp         = int(calendar.timegm(to_time.utctimetuple()))   * 1000
        requests.append(request)

    def fetch_next(index):
        if index >= len(requests):
            print("\nAll chunks fetched")
            reactor.stop()
            return
        deferred = client.send(requests[index])
        def on_success(result):
            trendbars = Protobuf.extract(result)
            barsData = list(map(transformTrendbar, trendbars.trendbar))
            global dailyBars
            dailyBars.extend(barsData)
            print(f"\nFetched chunk {index+1}/{len(requests)}, bars: {len(barsData)}")
            fetch_next(index + 1)
        deferred.addCallbacks(on_success, onError)

    global dailyBars
    dailyBars.clear()
    fetch_next(0)

def accountAuthResponseCallback(result):
    print("\nAccount authenticated")
    request = ProtoOASymbolsListReq()
    request.ctidTraderAccountId = credentials["AccountId"]
    request.includeArchivedSymbols = False
    deferred = client.send(request)
    deferred.addCallbacks(symbolsResponseCallback, onError)

def applicationAuthResponseCallback(result):
    print("\nApplication authenticated")
    request = ProtoOAAccountAuthReq()
    request.ctidTraderAccountId = credentials["AccountId"]
    request.accessToken = credentials["AccessToken"]
    deferred = client.send(request)
    deferred.addCallbacks(accountAuthResponseCallback, onError)

def onError(failure):
    print("\nMessage Error:", failure)

def disconnected(client, reason):
    print("\nDisconnected:", reason)

def onMessageReceived(client, message):
    if message.payloadType in [
        ProtoHeartbeatEvent().payloadType,
        ProtoOAAccountAuthRes().payloadType,
        ProtoOAApplicationAuthRes().payloadType,
        ProtoOASymbolsListRes().payloadType,
        ProtoOAGetTrendbarsRes().payloadType,
    ]:
        return
    print("\nMessage received:\n", Protobuf.extract(message))

def connected(client):
    print("\nConnected")
    request = ProtoOAApplicationAuthReq()
    request.clientId     = credentials["ClientId"]
    request.clientSecret = credentials["Secret"]
    deferred = client.send(request)
    deferred.addCallbacks(applicationAuthResponseCallback, onError)

client.setConnectedCallback(connected)
client.setDisconnectedCallback(disconnected)
client.setMessageReceivedCallback(onMessageReceived)

In [7]:
client.startService()
reactor.run()


Connected

Application authenticated

Account authenticated

Symbols received

Fetched chunk 1/520, bars: 6431

Fetched chunk 2/520, bars: 6882

Fetched chunk 3/520, bars: 6882

Fetched chunk 4/520, bars: 6891

Fetched chunk 5/520, bars: 6952

Fetched chunk 6/520, bars: 6889

Fetched chunk 7/520, bars: 6804

Fetched chunk 8/520, bars: 6729

Fetched chunk 9/520, bars: 6892

Fetched chunk 10/520, bars: 6877

Fetched chunk 11/520, bars: 6806

Fetched chunk 12/520, bars: 6708

Fetched chunk 13/520, bars: 6832

Fetched chunk 14/520, bars: 5334

Fetched chunk 15/520, bars: 5210

Fetched chunk 16/520, bars: 6889

Fetched chunk 17/520, bars: 6882

Fetched chunk 18/520, bars: 6850

Fetched chunk 19/520, bars: 5772

Fetched chunk 20/520, bars: 6892

Fetched chunk 21/520, bars: 6889

Fetched chunk 22/520, bars: 6890

Fetched chunk 23/520, bars: 6821

Fetched chunk 24/520, bars: 6867

Fetched chunk 25/520, bars: 6892

Fetched chunk 26/520, bars: 6801

Fetched chunk 27/520, bars: 6805

Fetched chu

In [8]:
df = pd.DataFrame(
    np.array(dailyBars),
    columns=['Time', 'Open', 'High', 'Low', 'Close', 'Volume']
).drop_duplicates().reset_index(drop=True)

for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    df[col] = pd.to_numeric(df[col])

In [9]:
df['Time'].describe()

count                             3214759
mean     2021-07-29 07:24:55.632979+00:00
min             2016-04-18 20:35:00+00:00
25%             2019-04-25 15:59:30+00:00
50%             2021-08-31 09:38:00+00:00
75%             2023-12-07 06:43:30+00:00
max             2026-04-06 20:34:00+00:00
Name: Time, dtype: object

In [10]:
df = df.sort_values('Time').drop_duplicates().reset_index(drop=True)
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows to {output_path}")

Saved 3214759 rows to ../../data/US500_M1_520weeks.csv


In [11]:
df

,Time,Open,High,Low,Close,Volume
0,2016-04-18 20:35:00+00:00,2092.6,2092.6,2092.5,2092.6,6
1,2016-04-18 20:36:00+00:00,2092.5,2092.5,2092.5,2092.5,2
2,2016-04-18 20:38:00+00:00,2092.3,2092.3,2092.2,2092.2,4
3,2016-04-18 20:41:00+00:00,2092.4,2092.6,2092.4,2092.6,4
4,2016-04-18 20:42:00+00:00,2092.5,2092.7,2092.5,2092.7,6
...,...,...,...,...,...,...
3214754,2026-04-06 20:30:00+00:00,6610.5,6611.8,6610.5,6611.8,30
3214755,2026-04-06 20:31:00+00:00,6611.6,6611.8,6610.8,6611.0,38
3214756,2026-04-06 20:32:00+00:00,6611.3,6611.3,6610.5,6610.8,36
3214757,2026-04-06 20:33:00+00:00,6610.9,6611.3,6610.5,6610.8,30
